In [87]:
import pandas as pd
import numpy as np
import json
import time
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from collections import Counter
from sklearn.utils.class_weight import compute_class_weight

In [ ]:
class CFG:
    # данные
    DATA_PATH = "/Users/test/Desktop/GP5/DL_project_5/data/nlp_dataset.csv" 
    TEXT_COL  = "clean_text"  
    TARGET    = "Product"   

    # DagsHub MLflow
    DAGSHUB_OWNER = "kulikanton05"
    DAGSHUB_REPO  = "GP5"

    TEST_SIZE    = 0.2  
    VAL_SIZE     = 0.1   
    RANDOM_STATE = 42 

    # словарь и последовательности для RNN
    MAX_LEN   = 300
    MIN_FREQ  = 5
    MAX_VOCAB = 20000

    # обучение RNN
    EMBED_DIM  = 128
    HIDDEN_DIM = 128
    NUM_LAYERS = 1
    DROPOUT    = 0.3
    RNN_EPOCHS = 8
    RNN_BATCH  = 64
    RNN_LR     = 1e-3
    CLIP_NORM  = 1.0

    # обучение DistilBERT
    BERT_MODEL  = "distilbert-base-uncased"
    BERT_MAXLEN = 128
    BERT_EPOCHS = 2
    BERT_BATCH  = 8
    BERT_LR     = 2e-5
    
def seed_everything(seed=CFG.RANDOM_STATE):
    np.random.seed(seed)
    torch.manual_seed(seed)

seed_everything()

In [20]:
df = pd.read_csv(CFG.DATA_PATH)
df

,Consumer complaint narrative,Product,Issue,word_count,clean_text
0,I mobile deposited a XXXX check to Ameris Bank...,Checking or savings account,Managing an account,188,i mobile deposited a check to ameris bank on y...
1,"Credence Resource Management , LLC ( Use the a...",Debt collection,Attempts to collect debt not owed,332,credence resource management llc use the addre...
2,"Help my rights are being violated, 15 U.S.C. 1...",Debt collection,Took or threatened to take negative or legal a...,85,help my rights are being violated u s c g b of...
3,This company is violating my rights they have ...,Debt collection,Took or threatened to take negative or legal a...,74,this company is violating my rights they have ...
4,My information was heisted and this accounts a...,Credit card,Other,31,my information was heisted and this accounts a...
...,...,...,...,...,...
53893,My credit reports are inaccurate. These inaccu...,Credit reporting or other personal consumer re...,Incorrect information on your report,39,my credit reports are inaccurate these inaccur...
53894,I received a very shady voicemail from structu...,Debt collection,Other,39,i received a very shady voicemail from structu...
53895,In XXXX of XXXX I was charged for a Best Buy r...,Credit card,Problem with a purchase shown on your statement,434,in of i was charged for a best buy renewal of ...
53896,"My name is XXXX XXXX, residing at XXXX XXXX XX...",Debt collection,Attempts to collect debt not owed,277,my name is residing at ga my date of birth is ...


In [91]:
df['Issue'].value_counts()

Issue
Other                                                           15362
Attempts to collect debt not owed                               12106
Managing an account                                              6592
Written notification about debt                                  3973
Problem with a purchase shown on your statement                  3915
False statements or representation                               3119
Took or threatened to take negative or legal action              2049
Incorrect information on your report                             1911
Problem with a lender or other company charging your account     1748
Closing an account                                               1597
Fraud or scam                                                    1526
Name: count, dtype: int64

In [21]:
le = LabelEncoder()
y = le.fit_transform(df[CFG.TARGET])
X = df[CFG.TEXT_COL]

In [22]:
dict(zip(le.classes_, le.transform(le.classes_)))

{'Checking or savings account': np.int64(0),
 'Credit card': np.int64(1),
 'Credit reporting or other personal consumer reports': np.int64(2),
 'Debt collection': np.int64(3),
 'Money transfer, virtual currency, or money service': np.int64(4)}

In [55]:
class_names = list(le.classes_)
num_classes = len(class_names)
class_names, num_classes

(['Checking or savings account',
  'Credit card',
  'Credit reporting or other personal consumer reports',
  'Debt collection',
  'Money transfer, virtual currency, or money service'],
 5)

In [23]:
X_trv, X_test, y_trv, y_test = train_test_split(X, y, test_size=CFG.TEST_SIZE, random_state=CFG.RANDOM_STATE, stratify=y)
val_rel = CFG.VAL_SIZE/ (1.0 - CFG.TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(X_trv, y_trv, test_size=val_rel, random_state=CFG.RANDOM_STATE, stratify=y_trv)

In [24]:
len(X_train), len(X_val), len(X_test)

(37728, 5390, 10780)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

mps


In [ ]:
def save_split_csv(X, y, part_name):
    out = pd.DataFrame({
        CFG.TEXT_COL: X,
        "label": y,
        CFG.TARGET: le.inverse_transform(y)
    })
    path = f'/Users/test/Desktop/GP5/DL_project_5/artifacts/{part_name}.csv'
    out.to_csv(path, index=False)
    return path

SPLIT_PATHS = {
    "train": save_split_csv(X_train, y_train, "train"),
    "val":   save_split_csv(X_val,   y_val,   "val"),
    "test":  save_split_csv(X_test,  y_test,  "test"),
}
print("Сохранены сплиты:", SPLIT_PATHS)

Сохранены сплиты: {'train': '/Users/test/Desktop/GP5/DL_project_5/artifacts/train.csv', 'val': '/Users/test/Desktop/GP5/DL_project_5/artifacts/val.csv', 'test': '/Users/test/Desktop/GP5/DL_project_5/artifacts/test.csv'}


In [ ]:
PAD_TOKEN, UNK_TOKEN = "<pad>", "<unk>"
PAD_IDX,   UNK_IDX   = 0, 1


def tokenize(text):
    return str(text).split()


def build_vocab(texts, min_freq=CFG.MIN_FREQ, max_vocab=CFG.MAX_VOCAB):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))

    kept = [w for w, c in counter.most_common() if c >= min_freq]
    kept = kept[: max_vocab - 2]          # -2 под спецтокены

    stoi = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for word in kept:
        stoi[word] = len(stoi)
    return stoi

def encode(text, stoi, max_len=CFG.MAX_LEN):
    '''Текст -> список индексов. Неизвестные слова -> UNK, длина <= max_len.'''
    ids = [stoi.get(tok, UNK_IDX) for tok in tokenize(text)][:max_len]
    if not ids:
        ids = [UNK_IDX]
    return ids

stoi = build_vocab(X_train)
vocab_size = len(stoi)
print("Размер словаря:", vocab_size)

Размер словаря: 11112


In [38]:
list(stoi.items())[:30]

[('<pad>', 0),
 ('<unk>', 1),
 ('the', 2),
 ('i', 3),
 ('to', 4),
 ('and', 5),
 ('a', 6),
 ('of', 7),
 ('my', 8),
 ('that', 9),
 ('this', 10),
 ('account', 11),
 ('was', 12),
 ('not', 13),
 ('on', 14),
 ('in', 15),
 ('is', 16),
 ('credit', 17),
 ('for', 18),
 ('or', 19),
 ('with', 20),
 ('they', 21),
 ('have', 22),
 ('debt', 23),
 ('me', 24),
 ('as', 25),
 ('it', 26),
 ('from', 27),
 ('am', 28),
 ('reporting', 29)]

In [39]:
list(stoi.items())[100:150]

[('inaccurate', 100),
 ('do', 101),
 ('claim', 102),
 ('sent', 103),
 ('money', 104),
 ('made', 105),
 ('written', 106),
 ('days', 107),
 ('proof', 108),
 ('also', 109),
 ('transactions', 110),
 ('one', 111),
 ('financial', 112),
 ('charges', 113),
 ('phone', 114),
 ('issue', 115),
 ('so', 116),
 ('unauthorized', 117),
 ('due', 118),
 ('charge', 119),
 ('there', 120),
 ('requested', 121),
 ('closed', 122),
 ('date', 123),
 ('alleged', 124),
 ('only', 125),
 ('through', 126),
 ('name', 127),
 ('will', 128),
 ('called', 129),
 ('notice', 130),
 ('xxxxxxxx', 131),
 ('contacted', 132),
 ('out', 133),
 ('required', 134),
 ('fdcpa', 135),
 ('fraudulent', 136),
 ('email', 137),
 ('call', 138),
 ('agreement', 139),
 ('back', 140),
 ('then', 141),
 ('disputed', 142),
 ('service', 143),
 ('identity', 144),
 ('immediately', 145),
 ('practices', 146),
 ('rights', 147),
 ('address', 148),
 ('letter', 149)]

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, stoi, max_len=CFG.MAX_LEN):
        self.encoded = [encode(t, stoi, max_len) for t in texts]
        self.labels  = np.asarray(labels, dtype=np.int64)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], int(self.labels[idx])

In [92]:
def collate_batch(batch):
    seqs, labels = zip(*batch)

    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.int64)
    max_len = int(lengths.max())
    padded = torch.full((len(seqs), max_len), PAD_IDX, dtype=torch.int64)
    for i, s in enumerate(seqs):
        padded[i, : len(s)] = torch.tensor(s, dtype=torch.int64)

    labels = torch.tensor(labels, dtype=torch.int64)
    return padded, lengths, labels

train_ds = TextDataset(X_train, y_train, stoi)
val_ds   = TextDataset(X_val,   y_val,   stoi)
test_ds  = TextDataset(X_test,  y_test,  stoi)

train_loader = DataLoader(train_ds, batch_size=CFG.RNN_BATCH, shuffle=True,  collate_fn=collate_batch)
val_loader   = DataLoader(val_ds,   batch_size=CFG.RNN_BATCH, shuffle=False, collate_fn=collate_batch)
test_loader  = DataLoader(test_ds,  batch_size=CFG.RNN_BATCH, shuffle=False, collate_fn=collate_batch)

xb, lb, yb = next(iter(train_loader))
print("padded:", xb.shape, "lengths:", lb.shape, "labels:", yb.shape)


padded: torch.Size([64, 300]) lengths: torch.Size([64]) labels: torch.Size([64])


In [ ]:
def compute_metrics(y_true, y_pred):
    return {
        "accuracy":        accuracy_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro":    recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro":        f1_score(y_true, y_pred, average="macro", zero_division=0),
    }


def train_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, n = 0.0, 0
    preds, trues = [], []

    for text, lengths, labels in loader:
        text, labels = text.to(device), labels.to(device)   # переносим батч на MPS

        optimizer.zero_grad()
        logits = model(text, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG.CLIP_NORM)  # стабилизируем RNN
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        n += labels.size(0)
        preds.extend(logits.argmax(1).detach().cpu().tolist())
        trues.extend(labels.detach().cpu().tolist())

    metrics = compute_metrics(trues, preds)
    metrics["loss"] = total_loss / max(n, 1)
    return metrics


@torch.no_grad()
def validate_epoch(model, loader, criterion):
    model.eval()
    total_loss, n = 0.0, 0
    preds, trues = [], []

    for text, lengths, labels in loader:
        text, labels = text.to(device), labels.to(device)
        logits = model(text, lengths)
        loss = criterion(logits, labels)

        total_loss += loss.item() * labels.size(0)
        n += labels.size(0)
        preds.extend(logits.argmax(1).detach().cpu().tolist())
        trues.extend(labels.detach().cpu().tolist())

    metrics = compute_metrics(trues, preds)
    metrics["loss"] = total_loss / max(n, 1)
    return metrics


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    preds, trues = [], []
    for text, lengths, labels in loader:
        text = text.to(device)
        logits = model(text, lengths)
        preds.extend(logits.argmax(1).detach().cpu().tolist())
        trues.extend(labels.tolist())
    return compute_metrics(trues, preds), trues, preds


In [47]:
import dagshub
import mlflow

dagshub.init(repo_owner=CFG.DAGSHUB_OWNER, repo_name=CFG.DAGSHUB_REPO, mlflow=True)
mlflow.set_experiment("Product_Classification")
print("Tracking URI:", mlflow.get_tracking_uri())

Accessing as kulikanton05

Initialized MLflow to track repo "kulikanton05/GP5"

Repository kulikanton05/GP5 initialized!

Tracking URI: https://dagshub.com/kulikanton05/GP5.mlflow


In [ ]:
def log_rnn_run(model, model_name, history, test_metrics, y_true, y_pred):
    with mlflow.start_run(run_name=model_name):
        # Параметры эксперимента
        mlflow.log_params({
            "model_name":   model_name,
            "target":       CFG.TARGET,
            "batch_size":   CFG.RNN_BATCH,
            "epochs":       CFG.RNN_EPOCHS,
            "learning_rate": CFG.RNN_LR,
            "embedding_dim": CFG.EMBED_DIM,
            "hidden_dim":   CFG.HIDDEN_DIM,
            "vocab_size":   vocab_size,
            "max_len":      CFG.MAX_LEN,
        })

        # Метрики по эпохам
        for ep, h in enumerate(history, start=1):
            mlflow.log_metrics({
                "train_loss":      h["train_loss"],
                "val_loss":        h["val_loss"],
                "val_accuracy":    h["val_accuracy"],
                "val_f1_macro":    h["val_f1_macro"],
            }, step=ep)

        # Итоговые метрики на тесте
        mlflow.log_metrics({
            "accuracy":        test_metrics["accuracy"],
            "precision_macro": test_metrics["precision_macro"],
            "recall_macro":    test_metrics["recall_macro"],
            "f1_macro":        test_metrics["f1_macro"],
        })

        # classification report как текстовый артефакт
        report = classification_report(y_true, y_pred, target_names=class_names, zero_division=0)
        rep_path = f"/Users/test/Desktop/GP5/DL_project_5/artifacts/{model_name}_report.txt"
        with open(rep_path, "w") as f:
            f.write(report)
        mlflow.log_artifact(rep_path, artifact_path="reports")

        # Артефакты для воспроизводимости: модель, LabelEncoder, словарь
        mlflow.pytorch.log_model(model, artifact_path="model")

        le_path = "/Users/test/Desktop/GP5/DL_project_5/artifacts/label_classes.json"
        with open(le_path, "w") as f:
            json.dump(class_names, f, ensure_ascii=False)
        mlflow.log_artifact(le_path, artifact_path="assets")

        vocab_path = "/Users/test/Desktop/GP5/DL_project_5/artifacts/vocab.json"
        with open(vocab_path, "w") as f:
            json.dump(stoi, f)
        mlflow.log_artifact(vocab_path, artifact_path="assets")

        # Данные эксперимента
        for p in SPLIT_PATHS.values():
            mlflow.log_artifact(p, artifact_path="data")

    print(f"[{model_name}] залогирован в MLflow.")


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim=CFG.EMBED_DIM, hidden_dim=CFG.HIDDEN_DIM, num_layers=CFG.NUM_LAYERS, dropout=CFG.DROPOUT, pad_idx=PAD_IDX):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, bidirectional=False, batch_first=True,)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, text, lengths):
        embedded = self.embedding(text)         
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        last_hidden = self.dropout(hidden[-1])              
        return self.fc(last_hidden)                       


In [68]:
seed_everything()
lstm_model = LSTMClassifier(vocab_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=CFG.RNN_LR)

lstm_history = []
for epoch in range(1, CFG.RNN_EPOCHS + 1):
    t0 = time.time()
    tr = train_epoch(lstm_model, train_loader, criterion, optimizer)
    va = validate_epoch(lstm_model, val_loader, criterion)
    lstm_history.append({
        "train_loss": tr["loss"], "val_loss": va["loss"],
        "val_accuracy": va["accuracy"], "val_f1_macro": va["f1_macro"],
    })
    print(f"[LSTM] epoch {epoch}/{CFG.RNN_EPOCHS} | {time.time()-t0:4.1f}s "
          f"| train_loss {tr['loss']:.4f} | val_loss {va['loss']:.4f} "
          f"| val_acc {va['accuracy']:.3f} | val_f1 {va['f1_macro']:.3f}")


[LSTM] epoch 1/8 | 246.2s | train_loss 0.9400 | val_loss 0.7166 | val_acc 0.734 | val_f1 0.546


KeyboardInterrupt: 

In [58]:
lstm_test, lstm_true, lstm_pred = evaluate_model(lstm_model, test_loader)
print("LSTM test metrics:", {k: round(v, 4) for k, v in lstm_test.items()})
print()
print(classification_report(lstm_true, lstm_pred, target_names=class_names, zero_division=0))

LSTM test metrics: {'accuracy': 0.8558, 'precision_macro': 0.813, 'recall_macro': 0.8058, 'f1_macro': 0.809}

                                                     precision    recall  f1-score   support

                        Checking or savings account       0.78      0.81      0.79      2334
                                        Credit card       0.83      0.83      0.83      2275
Credit reporting or other personal consumer reports       0.82      0.81      0.81       542
                                    Debt collection       0.95      0.94      0.94      4629
 Money transfer, virtual currency, or money service       0.69      0.63      0.66      1000

                                           accuracy                           0.86     10780
                                          macro avg       0.81      0.81      0.81     10780
                                       weighted avg       0.86      0.86      0.86     10780



In [59]:
log_rnn_run(lstm_model, "LSTM", lstm_history, lstm_test, lstm_true, lstm_pred)

2026/06/15 03:48:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 03:48:24 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run LSTM at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/0/runs/9c2838f4df4d48cd8216fdaba26dfb63
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/0
[LSTM] залогирован в MLflow.


In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim=CFG.EMBED_DIM, hidden_dim=CFG.HIDDEN_DIM, num_layers=CFG.NUM_LAYERS, dropout=CFG.DROPOUT, pad_idx=PAD_IDX):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, bidirectional=True, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, text, lengths):
        embedded = self.embedding(text)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        last_hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        last_hidden = self.dropout(last_hidden)
        return self.fc(last_hidden)


In [70]:
seed_everything()
bilstm_model = BiLSTMClassifier(vocab_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(bilstm_model.parameters(), lr=CFG.RNN_LR)

bilstm_history = []
for epoch in range(1, CFG.RNN_EPOCHS + 1):
    t0 = time.time()
    tr = train_epoch(bilstm_model, train_loader, criterion, optimizer)
    va = validate_epoch(bilstm_model, val_loader, criterion)
    bilstm_history.append({
        "train_loss": tr["loss"], "val_loss": va["loss"],
        "val_accuracy": va["accuracy"], "val_f1_macro": va["f1_macro"],
    })
    print(f"[BiLSTM] epoch {epoch}/{CFG.RNN_EPOCHS} | {time.time()-t0:4.1f}s "
          f"| train_loss {tr['loss']:.4f} | val_loss {va['loss']:.4f} "
          f"| val_acc {va['accuracy']:.3f} | val_f1 {va['f1_macro']:.3f}")


[BiLSTM] epoch 1/8 | 425.4s | train_loss 0.7488 | val_loss 0.5412 | val_acc 0.794 | val_f1 0.731
[BiLSTM] epoch 2/8 | 468.4s | train_loss 0.4797 | val_loss 0.4446 | val_acc 0.840 | val_f1 0.787
[BiLSTM] epoch 3/8 | 439.2s | train_loss 0.3904 | val_loss 0.4277 | val_acc 0.843 | val_f1 0.793
[BiLSTM] epoch 4/8 | 440.7s | train_loss 0.3294 | val_loss 0.4124 | val_acc 0.855 | val_f1 0.806
[BiLSTM] epoch 5/8 | 429.3s | train_loss 0.2741 | val_loss 0.4150 | val_acc 0.856 | val_f1 0.806
[BiLSTM] epoch 6/8 | 433.2s | train_loss 0.2217 | val_loss 0.4316 | val_acc 0.859 | val_f1 0.814
[BiLSTM] epoch 7/8 | 425.7s | train_loss 0.1693 | val_loss 0.4690 | val_acc 0.856 | val_f1 0.807
[BiLSTM] epoch 8/8 | 436.8s | train_loss 0.1288 | val_loss 0.5125 | val_acc 0.849 | val_f1 0.798


In [71]:
bilstm_test, bilstm_true, bilstm_pred = evaluate_model(bilstm_model, test_loader)
print("BiLSTM test metrics:", {k: round(v, 4) for k, v in bilstm_test.items()})
print()
print(classification_report(bilstm_true, bilstm_pred, target_names=class_names, zero_division=0))

BiLSTM test metrics: {'accuracy': 0.8481, 'precision_macro': 0.8031, 'recall_macro': 0.8028, 'f1_macro': 0.8023}

                                                     precision    recall  f1-score   support

                        Checking or savings account       0.77      0.79      0.78      2334
                                        Credit card       0.81      0.83      0.82      2275
Credit reporting or other personal consumer reports       0.83      0.87      0.85       542
                                    Debt collection       0.95      0.94      0.94      4629
 Money transfer, virtual currency, or money service       0.66      0.58      0.62      1000

                                           accuracy                           0.85     10780
                                          macro avg       0.80      0.80      0.80     10780
                                       weighted avg       0.85      0.85      0.85     10780



In [72]:
log_rnn_run(bilstm_model, "BiLSTM", bilstm_history, bilstm_test, bilstm_true, bilstm_pred)

2026/06/15 05:12:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 05:12:17 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run BiLSTM at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/0/runs/e77f3b76457a408e9509c8c8bdf91f54
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/0
[BiLSTM] залогирован в MLflow.


In [ ]:
class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim=CFG.EMBED_DIM, hidden_dim=CFG.HIDDEN_DIM, num_layers=CFG.NUM_LAYERS, dropout=CFG.DROPOUT, pad_idx=PAD_IDX):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=num_layers, bidirectional=False, batch_first=True,)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, text, lengths):
        embedded = self.embedding(text)
        packed = nn.utils.rnn.pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)

        last_hidden = self.dropout(hidden[-1])
        return self.fc(last_hidden)


In [74]:
seed_everything()
gru_model = GRUClassifier(vocab_size, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(gru_model.parameters(), lr=CFG.RNN_LR)

gru_history = []
for epoch in range(1, CFG.RNN_EPOCHS + 1):
    t0 = time.time()
    tr = train_epoch(gru_model, train_loader, criterion, optimizer)
    va = validate_epoch(gru_model, val_loader, criterion)
    gru_history.append({
        "train_loss": tr["loss"], "val_loss": va["loss"],
        "val_accuracy": va["accuracy"], "val_f1_macro": va["f1_macro"],
    })
    print(f"[GRU] epoch {epoch}/{CFG.RNN_EPOCHS} | {time.time()-t0:4.1f}s "
          f"| train_loss {tr['loss']:.4f} | val_loss {va['loss']:.4f} "
          f"| val_acc {va['accuracy']:.3f} | val_f1 {va['f1_macro']:.3f}")


[GRU] epoch 1/8 | 230.3s | train_loss 0.8185 | val_loss 0.5233 | val_acc 0.797 | val_f1 0.650
[GRU] epoch 2/8 | 231.4s | train_loss 0.4528 | val_loss 0.4336 | val_acc 0.842 | val_f1 0.773
[GRU] epoch 3/8 | 227.7s | train_loss 0.3568 | val_loss 0.4027 | val_acc 0.853 | val_f1 0.801
[GRU] epoch 4/8 | 228.4s | train_loss 0.3000 | val_loss 0.3984 | val_acc 0.857 | val_f1 0.810
[GRU] epoch 5/8 | 219.9s | train_loss 0.2490 | val_loss 0.3978 | val_acc 0.863 | val_f1 0.819
[GRU] epoch 6/8 | 227.2s | train_loss 0.2010 | val_loss 0.4447 | val_acc 0.858 | val_f1 0.818
[GRU] epoch 7/8 | 224.8s | train_loss 0.1560 | val_loss 0.4677 | val_acc 0.855 | val_f1 0.809
[GRU] epoch 8/8 | 261.5s | train_loss 0.1207 | val_loss 0.5479 | val_acc 0.849 | val_f1 0.805


In [75]:
gru_test, gru_true, gru_pred = evaluate_model(gru_model, test_loader)
print("GRU test metrics:", {k: round(v, 4) for k, v in gru_test.items()})
print()
print(classification_report(gru_true, gru_pred, target_names=class_names, zero_division=0))

GRU test metrics: {'accuracy': 0.8519, 'precision_macro': 0.8071, 'recall_macro': 0.8184, 'f1_macro': 0.812}

                                                     precision    recall  f1-score   support

                        Checking or savings account       0.80      0.77      0.79      2334
                                        Credit card       0.82      0.83      0.83      2275
Credit reporting or other personal consumer reports       0.83      0.83      0.83       542
                                    Debt collection       0.95      0.93      0.94      4629
 Money transfer, virtual currency, or money service       0.63      0.72      0.67      1000

                                           accuracy                           0.85     10780
                                          macro avg       0.81      0.82      0.81     10780
                                       weighted avg       0.86      0.85      0.85     10780



In [76]:
log_rnn_run(gru_model, "GRU", gru_history, gru_test, gru_true, gru_pred)

2026/06/15 14:59:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 15:00:01 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run GRU at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/0/runs/5afc4176053b4d4fb460367acc45d7e0
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/0
[GRU] залогирован в MLflow.


In [ ]:
class TextCNN(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim=128, num_filters=128, filter_sizes=(2, 3, 4, 5), dropout=0.5, pad_idx=0):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.embedding_dropout = nn.Dropout(0.2)
        self.convs = nn.ModuleList([nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=fs) for fs in filter_sizes])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(filter_sizes), num_classes)

    def forward(self, text, lengths=None):
        embedded = self.embedding(text)
        embedded = embedded.permute(0, 2, 1)

        conv_outputs = []
        for conv in self.convs:
            x = torch.relu(conv(embedded))
            x = torch.max(x, dim=2).values
            conv_outputs.append(x)

        x = torch.cat(conv_outputs, dim=1)
        x = self.dropout(x)

        return self.fc(x)

In [88]:
seed_everything()

cnn_model = TextCNN(
    vocab_size=vocab_size,
    num_classes=num_classes,
    embed_dim=200,
    num_filters=256,
    filter_sizes=(2, 3, 4, 5),
    dropout=0.5,
    pad_idx=PAD_IDX
).to(device)

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(cnn_model.parameters(), lr=CFG.RNN_LR, weight_decay=1e-4)

cnn_history = []

for epoch in range(1, CFG.RNN_EPOCHS + 1):
    t0 = time.time()

    tr = train_epoch(cnn_model, train_loader, criterion, optimizer)
    va = validate_epoch(cnn_model, val_loader, criterion)

    cnn_history.append({
        "train_loss": tr["loss"],
        "val_loss": va["loss"],
        "val_accuracy": va["accuracy"],
        "val_f1_macro": va["f1_macro"],
    })

    print(
        f"[TextCNN] epoch {epoch}/{CFG.RNN_EPOCHS} | {time.time()-t0:4.1f}s "
        f"| train_loss {tr['loss']:.4f} | val_loss {va['loss']:.4f} "
        f"| val_acc {va['accuracy']:.3f} | val_f1 {va['f1_macro']:.3f}"
    )

[TextCNN] epoch 1/8 | 128.5s | train_loss 0.8218 | val_loss 0.5473 | val_acc 0.811 | val_f1 0.773
[TextCNN] epoch 2/8 | 102.4s | train_loss 0.6034 | val_loss 0.4768 | val_acc 0.815 | val_f1 0.768
[TextCNN] epoch 3/8 | 102.1s | train_loss 0.5287 | val_loss 0.5069 | val_acc 0.852 | val_f1 0.808
[TextCNN] epoch 4/8 | 99.5s | train_loss 0.4576 | val_loss 0.4475 | val_acc 0.851 | val_f1 0.807
[TextCNN] epoch 5/8 | 100.3s | train_loss 0.4177 | val_loss 0.4300 | val_acc 0.863 | val_f1 0.821
[TextCNN] epoch 6/8 | 99.3s | train_loss 0.3845 | val_loss 0.4260 | val_acc 0.863 | val_f1 0.821
[TextCNN] epoch 7/8 | 99.5s | train_loss 0.3492 | val_loss 0.4200 | val_acc 0.867 | val_f1 0.823
[TextCNN] epoch 8/8 | 99.4s | train_loss 0.3172 | val_loss 0.4403 | val_acc 0.866 | val_f1 0.821


In [89]:
cnn_test, cnn_true, cnn_pred = evaluate_model(cnn_model, test_loader)

print("TextCNN test metrics:", {k: round(v, 4) for k, v in cnn_test.items()})
print()

print(classification_report(
    cnn_true,
    cnn_pred,
    target_names=class_names,
    zero_division=0
))

TextCNN test metrics: {'accuracy': 0.8654, 'precision_macro': 0.8081, 'recall_macro': 0.8421, 'f1_macro': 0.8218}

                                                     precision    recall  f1-score   support

                        Checking or savings account       0.79      0.85      0.82      2334
                                        Credit card       0.90      0.81      0.85      2275
Credit reporting or other personal consumer reports       0.71      0.91      0.80       542
                                    Debt collection       0.96      0.93      0.94      4629
 Money transfer, virtual currency, or money service       0.69      0.71      0.70      1000

                                           accuracy                           0.87     10780
                                          macro avg       0.81      0.84      0.82     10780
                                       weighted avg       0.87      0.87      0.87     10780



In [90]:
log_rnn_run(cnn_model, "TextCNN", cnn_history, cnn_test, cnn_true, cnn_pred)

2026/06/15 18:56:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/15 18:56:54 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.


🏃 View run TextCNN at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/0/runs/fce5437f249f497b87f48c4dd0ed6b85
🧪 View experiment at: https://dagshub.com/kulikanton05/GP5.mlflow/#/experiments/0
[TextCNN] залогирован в MLflow.
